In [1]:
!pip install -r /work/pi_elinor_broadinstitute_org/users/colin/pheno/requirements_fixed.txt

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import sys
import importlib
sys.path.append("/work/pi_elinor_broadinstitute_org/users/colin/pheno/")

In [3]:
import pheno
import pheno.phenodb
importlib.reload(pheno.phenodb)
from pheno.phenodb import PhenoDB

db_dog = PhenoDB("/scratch4/workspace/bryc_broadinstitute_org-darwins_cats/phenotype/2026-06-12", "dog")

INFO:pheno.phenodb:Verifying that all necessary cache files exist in path 
           /scratch4/workspace/bryc_broadinstitute_org-darwins_cats/phenotype/2026-06-12
INFO:pheno.phenodb:10 input files verified.
INFO:pheno.phenodb:Loading cache of data phenotype data
INFO:pheno.phenodb:Loading completed.
INFO:pheno.phenodb:Filtering and cleaning phenotype data:
INFO:pheno.phenodb:   141 animals removed for being test or fake. 50199 remaining
INFO:pheno.phenodb:   2006 deceased animals. Removing their ages
INFO:pheno.phenodb:   41 animals with ages above 30, excluding
INFO:pheno.phenodb:   Extracting breed names
INFO:pheno.phenodb:   Processing places table to extract location names
INFO:pheno.phenodb:Adding place ID information for all animal birth locations


In [4]:
dog_df = db_dog.get_phenotype_dataframe(source="all", all_animals=True)

INFO:pheno.phenodb:Loading data from all tables.
INFO:pheno.phenodb:Loaded 316044 data points from animal_survey_answers table
INFO:pheno.phenodb:Loaded 4198160 data points from animal_survey table
INFO:pheno.phenodb:Adding kit barcodes for each animal


In [5]:
import pandas as pd
import numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr
from pathlib import Path
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.model_selection import train_test_split


Path.cwd()

PosixPath('/home/cklick_broadinstitute_org')

This project uses the following vcf file: /project/pi_elinor_broadinstitute_org/darwins_dogs/darwins_dogs_gwas/low_pass_genetic_set_canfam4_2024/DarwinsDogs_2024_N-3277_canfam4.vcf.gz

A PLINK binary dataset was previously generated from this VCF, with genotype probability, biallelic-variant, and minor allele frequency filters reflected in its filename (gp-0.70_biallelic_maf0.01). The resulting dataset contained approximately 10 million variants.

Linkage disequilibrium (LD) pruning was performed on this dataset using PLINK 2:

plink2
--bfile /project/pi_elinor_broadinstitute_org/darwins_dogs/darwins_dogs_gwas/low_pass_genetic_set_canfam4_2024/DarwinsDogs_2024_N-3277_canfam4_gp-0.70_biallelic_maf0.01
--chr 1-38
--dog
--indep-pairwise 1000 100 0.2
--memory 12000
--out genome_prune_r0.2
--threads 4

LD threshold of (r^2=0.2) was used to identify a less-redundant set of variants. This resulted genome_prune_r0.2.prune.in, which contained 406058 SNPs.

Construction of the SNP selection list

The LD-pruned variant list, `genome_prune_r0.2.prune.in` (406,058 variants), was supplemented with previously identified dog body-size-associated variants derived from `cf3_to_cf4_body_size_sites.txt` in the Darwin's Dogs GitHub repository. The body-size variant positions were matched to variant IDs in the source PLINK .bim file, yielding `candidate_ids.txt`. These candidate IDs were combined with the LD-pruned variant IDs to create `keep_r0.2.txt` (408,230 unique variant IDs). The exact reason some original body-size positions were not represented in the final list remains to be verified.

The resulting selection list contained 408,230 variant IDs and was used to construct the panel_r0.2 PLINK dataset. The panel_r0.2 PLINK dataset was created with the following command:

--bfile /project/pi_elinor_broadinstitute_org/darwins_dogs/darwins_dogs_gwas/low_pass_genetic_set_canfam4_2024/DarwinsDogs_2024_N-3277_canfam4_gp-0.70_biallelic_maf0.01 --dog --extract keep_r0.2.txt --make-bed --out panel_r0.2

Subset dog_df so that it only contains genotyped dogs (fam["IID"])

In [6]:
fam = pd.read_csv("panel_r0.2.fam", sep=r"\s+", header=None,
                   names=["FID","IID","PAT","MAT","SEX","PHENO"])
genotyped_ids = fam["IID"]

dog_df_sub = dog_df.merge(genotyped_ids.to_frame("original_pet_id"),
                           on="original_pet_id", how="inner")
assert len(dog_df_sub) == 3277, f"got {len(dog_df_sub)}"

Subset the 3277 dogs to dogs with a non NA entry for 'age_at_profile_creation', 'sex', 'neutered', question '227', and an age >= 1. Map survey answers of question 227 to 'height_numeric' (1, 2, 3, 4, or 5).

In [7]:
height_map = {
    'Ankle high or shorter': 1,
    'Calf high': 2,
    'Knee high': 3,
    'Thigh high': 4,
    'Hip high or higher': 5,
}

analysis_df = dog_df_sub[
    dog_df_sub['age_at_profile_creation'].notna() &
    (dog_df_sub['age_at_profile_creation'] >= 1) &
    dog_df_sub['sex'].notna() &
    dog_df_sub['neutered'].notna() &
    dog_df_sub['227'].notna()
].copy()

analysis_df['height_numeric'] = analysis_df['227'].map(height_map)
analysis_df = analysis_df.dropna(subset=['height_numeric'])

print(len(analysis_df))

2539


Split 2539 dogs into training and test sets.

In [8]:
train_df, test_df = train_test_split(
    analysis_df,
    test_size=0.2,
    stratify=analysis_df['height_numeric'],
    random_state=42
)
print(len(train_df), len(test_df))

2031 508


Next create a text file that will be used as input for the BayesR command.  The text file has 'FID', 'IID', and 'height_numeric' columns.

In [9]:
pheno_train = train_df[['original_pet_id', 'height_numeric']].merge(
    fam[['IID']], left_on='original_pet_id', right_on='IID', how='inner'
)
assert len(pheno_train) == len(train_df), "ID mismatch — some train dogs not found in .fam"

pheno_train['FID'] = 0
pheno_train = pheno_train[['FID', 'IID', 'height_numeric']]
pheno_train.to_csv('height_pheno_train_only.txt', sep='\t', header=False, index=False)

Create a covariates file (sex, neuter status, and age) to be used as input for the BayesR command.

In [10]:
age_col = 'age_at_profile_creation'
cov = train_df[['original_pet_id', 'sex', 'neutered', age_col]].copy()

cov['sex_male'] = (cov['sex'].str.lower() == 'male').astype(float)
cov['neutered_yes'] = (cov['neutered'].astype(str).str.lower() == 'yes').astype(float)

assert set(cov['sex'].str.lower().unique()) <= {'male', 'female'}
assert set(cov['neutered'].astype(str).str.lower().unique()) <= {'yes', 'no'}

cov = cov.merge(fam[['IID']], left_on='original_pet_id', right_on='IID', how='inner')
assert len(cov) == len(train_df), "ID mismatch"
assert not cov[['sex_male', 'neutered_yes', age_col]].isna().any().any()

cov['FID'] = 0
cov_out = cov[['FID', 'IID', 'sex_male', 'neutered_yes', age_col]]
cov_out.to_csv('height_covar_train_only.txt', sep='\t', header=False, index=False)
print(len(cov_out))

2031


BayesR was used to estimate SNP effect sizes for dog height from the training dogs' genotypes and measured heights, while accounting for sex, neuter status, and age as covariates. It models SNP effects using a mixture of effect-size distributions, allowing most SNPs to have zero or small effects and a smaller number to have larger effects.

The command produces SNP-level results, including estimated allele effects (A1Effect), posterior probabilities for the four effect-size categories (Pi1–Pi4), and posterior inclusion probabilities (PIP). These estimates can be used to construct a polygenic score for each dog.

gctb
--bfile panel_r0.2
--pheno height_pheno_train_only.txt
--covar height_covar_train_only.txt
--bayes R
--pi 0.95,0.02,0.02,0.01
--gamma 0,0.01,0.1,1
--chain-length 10000
--burn-in 2000
--out panel_r0.2_height_trainonly
--thread 4

In [11]:
snp = pd.read_csv('panel_r0.2_height_trainonly.snpRes', sep=r'\s+')
# Create file to input into command calculating the scores
# snp[['Name', 'A1', 'A1Effect']].to_csv('height_bayesr_weights_trainonly.txt',
#                                        sep='\t', index=False, header=False)

Use the following command to generate the PGS score for all dogs (panel_r0.2_height_bayesr_pgs_trainonly.sscore).

plink2
--bfile panel_r0.2
--dog
--score height_bayesr_weights_trainonly.txt 1 2 3 cols=+scoresums
--out panel_r0.2_height_bayesr_pgs_trainonly

Create a wide-format admixture DataFrame (admix_wide) with one row per animal and one column per population, containing each animal’s estimated ancestry percentage for that population. Missing population percentages are filled with zero.

In [12]:
admix_latest = db_dog.admixture_results.sort_values('timestamp').drop_duplicates(subset='animal_id', keep='last')
key_lookup = {str(k): v for k, v in db_dog.populations.set_index('id')['key'].to_dict().items()}
name_lookup = db_dog.populations.set_index('id')['name'].to_dict()
exploded = admix_latest[['animal_id', 'populations']].explode('populations').reset_index(drop=True)
exploded = pd.concat(
    [exploded.drop(columns=['populations']),
     exploded['populations'].apply(pd.Series)],
    axis=1
)
exploded['population_key'] = exploded['population_id'].astype(str).map(key_lookup)
exploded_clean = exploded.dropna(subset=['population_id'])
admix_wide = exploded_clean.pivot_table(
    index='animal_id',
    columns='population_key',
    values='percentage',
    aggfunc='first'
).fillna(0).reset_index()

Add original_pet_id to admix_wide by mapping animal_id, then drop dogs that cannot be mapped.

In [13]:
# Translate animal_id -> original_pet_id so the merge key matches `merged`
admix_wide_renamed = admix_wide.rename(columns={'animal_id': 'original_pet_id_mapped'})

# admix_wide's 'animal_id' IS dog_df's index — so map it through the same lookup direction
animalid_to_petid = dog_df.reset_index().set_index('index')['original_pet_id']
admix_wide['original_pet_id'] = admix_wide['animal_id'].map(animalid_to_petid)

# Drop rows where mapping failed (dog not in dog_df at all)
print(admix_wide['original_pet_id'].isna().sum(), "unmapped after animal_id -> original_pet_id")
admix_wide = admix_wide.dropna(subset=['original_pet_id'])

27 unmapped after animal_id -> original_pet_id


Create pgs dataframe, consisting of columns 'IID' and 'pgs_bayesr'

In [14]:
for d in (train_df, test_df):
    d['IID'] = pd.to_numeric(d['original_pet_id']).astype('int64').astype(str)
# PGS scores
sc = pd.read_csv('panel_r0.2_height_bayesr_pgs_trainonly.sscore', sep='\t')
sc['IID'] = sc['IID'].astype(str)
pgs = sc[['IID', 'SCORE1_SUM']].rename(columns={'SCORE1_SUM': 'pgs_bayesr'})

Create admix_pet that contains 'IID' and the admixture columns.

In [15]:
# --- 1. admixture columns (admix_wide comes from your existing admixture code)
admix_cols = [c for c in admix_wide.columns if c not in ('animal_id', 'original_pet_id')]
admix_pet = admix_wide[['original_pet_id'] + admix_cols].copy()
admix_pet['IID'] = pd.to_numeric(admix_pet['original_pet_id']).astype('int64').astype(str)
admix_pet = admix_pet.drop(columns='original_pet_id')

Merge pgs and admix_pet on to train_final and test_final.

In [16]:
train_final = train_df.merge(pgs, on='IID', how='left').merge(admix_pet, on='IID', how='left')
test_final  = test_df.merge(pgs, on='IID', how='left').merge(admix_pet, on='IID', how='left')

Create covariate columns.

In [17]:
# covariate columns
for d in (train_final, test_final):
    d['sex_male'] = (d['sex'].str.lower() == 'male').astype(float)
    d['neutered_yes'] = (d['neutered'].astype(str).str.lower() == 'yes').astype(float)
    d['age'] = d['age_at_profile_creation']
cov = ['sex_male', 'neutered_yes', 'age']

In [18]:
# drop dogs missing anything, so every model uses the same dogs
need = admix_cols + cov + ['pgs_bayesr', 'height_numeric']
print("missing before drop:", train_final[need].isna().any(axis=1).sum(), "train;",
      test_final[need].isna().any(axis=1).sum(), "test")
train_final = train_final.dropna(subset=need).copy()
test_final  = test_final.dropna(subset=need).copy()
print(len(train_final), len(test_final))

missing before drop: 6 train; 1 test
2025 507


**Table 1 code.** Fits ancestry models on training dogs only, then uses 10-fold cross-validation within the test dogs to fit the few weights that combine PGS, covariates, and ancestry. It reports out-of-fold R² for five models (admixture, admixture+covariates, PGS, PGS+covariates, PGS+covariates+admixture) and bootstrap CIs for the key ΔR² comparisons.

In [19]:
y = test_final['height_numeric'].values
ytr = train_final['height_numeric'].values

# Ancestry features: fit on TRAIN only, applied to test
m_adm = LinearRegression().fit(train_final[admix_cols], ytr)
m_base = LinearRegression().fit(train_final[admix_cols + cov], ytr)
test_final['admix_pred'] = m_adm.predict(test_final[admix_cols])
test_final['base_pred'] = m_base.predict(test_final[admix_cols + cov])

# Models to compare (columns fed to the CV step)
sets_cv = {
    'Admixture':                  ['admix_pred'],
    'Admixture+covariates':       ['base_pred'],
    'PGS':                        ['pgs_bayesr'],
    'PGS+covariates':             ['pgs_bayesr'] + cov,
    'PGS+covariates+admixture':   ['pgs_bayesr'] + cov + ['admix_pred'],
}

# 10-fold CV within the test dogs (same folds for every model)
cv = KFold(10, shuffle=True, random_state=42)
preds_cv = {}
for k, cols in sets_cv.items():
    preds_cv[k] = cross_val_predict(LinearRegression(), test_final[cols], y, cv=cv)

table = pd.DataFrame({'R2': {k: round(r2_score(y, p), 3) for k, p in preds_cv.items()}})
print(table)

# Bootstrap over test dogs for differences in out-of-fold R2
def boot_cv(full, reduced, n=2000, seed=42):
    rng = np.random.default_rng(seed)
    a, b = preds_cv[full], preds_cv[reduced]
    d = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        d.append(r2_score(y[i], a[i]) - r2_score(y[i], b[i]))
    point = r2_score(y, a) - r2_score(y, b)
    lo, hi = np.percentile(d, [2.5, 97.5])
    return round(point, 3), (round(lo, 3), round(hi, 3))

print("PGS vs Admixture:                ", boot_cv('PGS', 'Admixture'))
print("PGS+cov vs Admixture+cov:        ", boot_cv('PGS+covariates', 'Admixture+covariates'))
print("Admixture added to PGS+cov:      ", boot_cv('PGS+covariates+admixture', 'PGS+covariates'))

                             R2
Admixture                 0.524
Admixture+covariates      0.549
PGS                       0.588
PGS+covariates            0.601
PGS+covariates+admixture  0.604
PGS vs Admixture:                 (0.064, (np.float64(0.016), np.float64(0.122)))
PGS+cov vs Admixture+cov:         (0.051, (np.float64(0.009), np.float64(0.096)))
Admixture added to PGS+cov:       (0.003, (np.float64(-0.007), np.float64(0.012)))


In [20]:
known = pd.DataFrame([
    ('IGF1',     15, 41495479, 41567874),
    ('GHR',       4, 67943814, 68083513),
    ('LCORL',     3, 91736911, 91864203),
    ('HMGA2',    10,  8704143,  8845709),
    ('IGF2BP2',  34, 18539585, 18694844),
    ('IGF1R',     3, 42299559, 42474531),
    ('SMAD2',     7, 43711952, 43761753),
    ('MSRB3',    10,  8250421,  8395405),
    ('MED13L',   26, 13102597, 13397930),
    ('ANAPC1',   17, 36389408, 36490398),
    ('SAR1B',    11, 22727640, 22737203),
], columns=['gene', 'chrom', 'start', 'end'])

PAD = 500_000
in_window = pd.Series(False, index=snp.index)
for _, k in known.iterrows():
    in_window |= (snp['Chrom'] == k['chrom']) & snp['Position'].between(k['start'] - PAD, k['end'] + PAD)
snp['known_500kb'] = in_window
print(snp['known_500kb'].sum(), "SNPs in 500 kb known-locus windows")

wcols = ['Name', 'A1', 'A1Effect']
snp.loc[snp['known_500kb'],  wcols].to_csv('height_bayesr_weights_known_500kb.txt', sep='\t', header=False, index=False)
snp.loc[~snp['known_500kb'], wcols].to_csv('height_bayesr_weights_rest_500kb.txt',  sep='\t', header=False, index=False)

3152 SNPs in 500 kb known-locus windows


In [21]:
# Load the 500 kb scores (drop-then-merge so re-running is safe)
for name, f in [('pgs_known_500kb', 'panel_r0.2_pgs_known_500kb.sscore'),
                ('pgs_rest_500kb',  'panel_r0.2_pgs_rest_500kb.sscore')]:
    sc = pd.read_csv(f, sep='\t')
    sc['IID'] = sc['IID'].astype(str)
    sc = sc[['IID', 'SCORE1_SUM']].rename(columns={'SCORE1_SUM': name})
    train_final = train_final.drop(columns=name, errors='ignore').merge(sc, on='IID', how='left')
    test_final  = test_final.drop(columns=name, errors='ignore').merge(sc, on='IID', how='left')

**Table 2 code** Residual correlation with height by PGS component

In [22]:
import numpy as np, pandas as pd
import statsmodels.api as sm
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
def resid(col):
    m = LinearRegression().fit(train_final[base], train_final[col])
    return test_final[col] - m.predict(test_final[base])

base = admix_cols + cov
h_res = resid('height_numeric')
print()
for c in ['pgs_bayesr', 'pgs_known_500kb', 'pgs_rest_500kb']:
    r, p = pearsonr(resid(c), h_res)
    print(f"{c:16s} r={r:.3f}  r^2={r**2:.3f}  p={p:.2g}")



pgs_bayesr       r=0.375  r^2=0.141  p=2.2e-18
pgs_known_500kb  r=0.353  r^2=0.125  p=2.4e-16
pgs_rest_500kb   r=0.197  r^2=0.039  p=8.2e-06


**Table 3 code**

In [23]:
names = ['p_known', 'p_top1k', 'p_next9k', 'p_bulk']
for g in names:
    sc = pd.read_csv(f'panel_r0.2_pgs_{g}.sscore', sep='\t')
    sc['IID'] = sc['IID'].astype(str)
    sc = sc[['IID', 'SCORE1_SUM']].rename(columns={'SCORE1_SUM': f'pgs_{g}'})
    train_final = train_final.drop(columns=f'pgs_{g}', errors='ignore').merge(sc, on='IID', how='left')
    test_final  = test_final.drop(columns=f'pgs_{g}', errors='ignore').merge(sc, on='IID', how='left')

# partition check: the four scores should sum to the full PGS
tot = sum(train_final[f'pgs_{g}'] for g in names)
print("corr with full PGS:", np.corrcoef(tot, train_final['pgs_bayesr'])[0, 1],
      "| max abs diff:", (tot - train_final['pgs_bayesr']).abs().max())   # want ~1.0 and ~0

base = admix_cols + cov
def resid(col):
    m = LinearRegression().fit(train_final[base], train_final[col])
    return test_final[col] - m.predict(test_final[base])

h_res = resid('height_numeric')
rng = np.random.default_rng(42)
n = len(h_res)
for g in names:
    x = resid(f'pgs_{g}')
    r, p = pearsonr(x, h_res)
    boots = [pearsonr(x.values[i], h_res.values[i])[0] for i in (rng.integers(0, n, n) for _ in range(1000))]
    tr_r = train_final[f'pgs_{g}'].corr(train_final['height_numeric'])
    print(f"{g:9s} r={r:.3f} CI [{np.percentile(boots, 2.5):.3f}, {np.percentile(boots, 97.5):.3f}] "
          f"p={p:.2g} | raw train r={tr_r:.2f}")

corr with full PGS: 0.9999999999986504 | max abs diff: 5.000000000254801e-06
p_known   r=0.353 CI [0.238, 0.461] p=2.4e-16 | raw train r=0.73
p_top1k   r=0.160 CI [0.059, 0.260] p=0.00029 | raw train r=0.80
p_next9k  r=0.155 CI [0.062, 0.244] p=0.00047 | raw train r=0.84
p_bulk    r=0.150 CI [0.070, 0.231] p=0.0007 | raw train r=0.85


**Table 4 code** The following is the Shapley analysis for Table 4

In [24]:
def r2_ls(h, cols):
    X = np.column_stack([np.ones(len(h))] + cols)
    beta = np.linalg.lstsq(X, h, rcond=None)[0]
    sse = ((h - X @ beta) ** 2).sum()
    sst = ((h - h.mean()) ** 2).sum()
    return 1 - sse / sst

def shapley_split(h, xk, xr):
    r_k, r_r, r_both = r2_ls(h, [xk]), r2_ls(h, [xr]), r2_ls(h, [xk, xr])
    s_k = 0.5 * (r_k + (r_both - r_r))
    s_r = 0.5 * (r_r + (r_both - r_k))
    return dict(r2_both=r_both, share_known=s_k, share_rest=s_r,
                pct_known=100 * s_k / r_both, pct_rest=100 * s_r / r_both,
                unique_known=r_both - r_r, unique_rest=r_both - r_k,
                shared=r_k + r_r - r_both)

h  = h_res.values
xk = resid('pgs_known_500kb').values
xr = resid('pgs_rest_500kb').values

est = shapley_split(h, xk, xr)
print({k: round(v, 3) for k, v in est.items()})

rng = np.random.default_rng(42)
n = len(h)
boots = [shapley_split(h[i], xk[i], xr[i])
         for i in (rng.integers(0, n, n) for _ in range(2000))]
for k in ['pct_known', 'pct_rest', 'share_known', 'share_rest']:
    vals = [b[k] for b in boots]
    print(f"{k:12s} {est[k]:.3f}  95% CI [{np.percentile(vals, 2.5):.3f}, {np.percentile(vals, 97.5):.3f}]")

{'r2_both': np.float64(0.15), 'share_known': np.float64(0.118), 'share_rest': np.float64(0.032), 'pct_known': np.float64(78.708), 'pct_rest': np.float64(21.292), 'unique_known': np.float64(0.111), 'unique_rest': np.float64(0.025), 'shared': np.float64(0.014)}
pct_known    78.708  95% CI [58.783, 92.236]
pct_rest     21.292  95% CI [7.764, 41.217]
share_known  0.118  95% CI [0.057, 0.192]
share_rest   0.032  95% CI [0.010, 0.065]


To quantify how breed-driven the polygenic score (PGS) is BayesR PGS was regressed on admixture and covariates, using ordinary least-squares linear regression. The model was fit on the training dogs only. R² in the held-out test dogs was 0.829.

In [25]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X_cols = admix_cols + cov

Xtr = train_final[X_cols].values
Xte = test_final[X_cols].values
ytr = train_final['pgs_bayesr'].values
yte = test_final['pgs_bayesr'].values

m = LinearRegression().fit(Xtr, ytr)

print(f"Train R2 = {r2_score(ytr, m.predict(Xtr)):.3f}")
print(f"Test  R2 = {r2_score(yte, m.predict(Xte)):.3f}")

Train R2 = 0.736
Test  R2 = 0.829
